# Solar PV Business Forecast Model

Build a monthly forecast model for a Solar PV business in Australia (July 2013 - June 2023).

In [ ]:
import pandas as pd
import numpy as np
import openpyxl
from calendar import monthrange
from datetime import datetime

# Load workbook and explore structure
wb = openpyxl.load_workbook('/workspace/data/MO13-Energy-Operations-Workbook.xlsx', data_only=True)
print('Sheet names:', wb.sheetnames)

# Dump all sheets fully so we can understand the layout
for name in wb.sheetnames:
    ws = wb[name]
    print(f'\n=== Sheet: {name} (rows={ws.max_row}, cols={ws.max_column}) ===')
    for row_idx in range(1, min(ws.max_row + 1, 80)):
        row_data = []
        for col_idx in range(1, min(ws.max_column + 1, 40)):
            val = ws.cell(row=row_idx, column=col_idx).value
            if val is not None:
                row_data.append(f'R{row_idx}C{col_idx}={repr(val)}')
        if row_data:
            print('  ' + ' | '.join(row_data))

In [ ]:
# Extract installation schedule and exchange rates from workbook
# Direct extraction from known data layout (Assumptions sheet)

import openpyxl
from datetime import datetime

wb = openpyxl.load_workbook('/workspace/data/MO13-Energy-Operations-Workbook.xlsx', data_only=True)
ws = wb['Assumptions']

small_systems = {}
large_systems = {}
exchange_rates = {}

# Data starts at row 37, columns: C3=date, C4=small, C5=large, C6=AUD:USD rate
for r in range(37, 200):
    date_val = ws.cell(row=r, column=3).value
    if date_val is None:
        break
    if isinstance(date_val, datetime):
        ym = (date_val.year, date_val.month)
    else:
        continue
    
    small_val = ws.cell(row=r, column=4).value
    large_val = ws.cell(row=r, column=5).value
    fx_val = ws.cell(row=r, column=6).value  # 1 AUD = X USD
    
    if small_val is not None:
        small_systems[ym] = int(small_val)
    if large_val is not None:
        large_systems[ym] = int(large_val)
    if fx_val is not None and isinstance(fx_val, (int, float)):
        exchange_rates[ym] = float(fx_val)

print(f'Extracted: {len(small_systems)} small, {len(large_systems)} large, {len(exchange_rates)} FX rates')
for ym in sorted(small_systems.keys())[:5]:
    print(f'  {ym}: small={small_systems[ym]}, large={large_systems[ym]}, FX={exchange_rates.get(ym, "N/A")}')

In [ ]:
# =================================================================
# CONSTANTS AND MODEL SETUP
# =================================================================

SMALL_COST_USD = 5000       # USD per small system
LARGE_COST_USD = 8000       # USD per large system
SMALL_CAPACITY_KW = 3       # kW per small system
LARGE_CAPACITY_KW = 5       # kW per large system
INSTALL_COST_AUD = 500      # AUD per system installation
GENERATION_RATE = 4         # kWh per kW per day (when new)
TARIFF_A_INITIAL = 0.25     # AUD per kWh
TARIFF_B = 0.08             # AUD per kWh (constant)
TARIFF_A_ESCALATION = 0.02  # 2% annual escalation on 1 July
SMALL_HOUSE_DAILY_KWH = 10  # kWh/day consumed at Tariff A per small system house
LARGE_HOUSE_DAILY_KWH = 15  # kWh/day consumed at Tariff A per large system house

# Monthly degradation: efficiency after 12 months = 99% of current
# monthly_retention^12 = 0.99  =>  monthly_retention = 0.99^(1/12)
monthly_retention = 0.99 ** (1.0 / 12.0)
print(f'Monthly retention factor: {monthly_retention:.12f}')
print(f'Verification: retention^12 = {monthly_retention**12:.10f} (should be 0.99)')

# Build timeline: July 2013 to June 2023
months = []
for y in range(2013, 2024):
    for m in range(1, 13):
        if y == 2013 and m < 7:
            continue
        if y == 2023 and m > 6:
            continue
        months.append((y, m))

days_in_month = {(y, m): monthrange(y, m)[1] for y, m in months}

# Installation period: July 2013 to June 2015
install_months = []
for y in range(2013, 2016):
    for m in range(1, 13):
        if y == 2013 and m < 7:
            continue
        if y == 2015 and m > 6:
            continue
        install_months.append((y, m))

# Fill missing months with 0 installations
for ym in install_months:
    small_systems.setdefault(ym, 0)
    large_systems.setdefault(ym, 0)

print(f'\nInstallation schedule ({len(install_months)} months):')
total_small = total_large = 0
for ym in install_months:
    s = small_systems[ym]
    l = large_systems[ym]
    total_small += s
    total_large += l
    print(f'  {ym[0]}-{ym[1]:02d}: small={s}, large={l}, cumul_small={total_small}, cumul_large={total_large}')

print(f'\nTotal systems: small={total_small}, large={total_large}')
print(f'Total capacity: {total_small * SMALL_CAPACITY_KW + total_large * LARGE_CAPACITY_KW} kW')

In [ ]:
# =================================================================
# HELPER FUNCTIONS
# =================================================================

def month_idx(ym):
    """Linear month index for difference calculations."""
    return ym[0] * 12 + ym[1]

def get_tariff_a(year, month):
    """Tariff A price for a given month."""
    # Escalation happens each 1 July, starting 1 July 2014
    # July 2013 - June 2014: 0 escalations
    # July 2014 - June 2015: 1 escalation
    # ...
    if year < 2014 or (year == 2014 and month < 7):
        n = 0
    else:
        fiscal_year_start = year if month >= 7 else year - 1
        n = fiscal_year_start - 2013
    return TARIFF_A_INITIAL * (1 + TARIFF_A_ESCALATION) ** n

def get_efficiency(months_since_install):
    """Panel efficiency after given months since installation."""
    if months_since_install <= 0:
        return 1.0
    return monthly_retention ** months_since_install

# =================================================================
# QUESTION 1: Cumulative rated capacity as at 15 September 2014
# =================================================================
cum_s = cum_l = 0
for ym in install_months:
    if ym <= (2014, 9):
        cum_s += small_systems[ym]
        cum_l += large_systems[ym]

total_capacity_sep2014 = cum_s * SMALL_CAPACITY_KW + cum_l * LARGE_CAPACITY_KW
print(f'Q1: Cumulative small={cum_s}, large={cum_l}')
print(f'Q1: Total rated capacity as at 15 Sep 2014 = {total_capacity_sep2014} kW')

options_q1 = {'A': 4000, 'B': 5000, 'C': 5500, 'D': 6000}
q1 = min(options_q1, key=lambda k: abs(options_q1[k] - total_capacity_sep2014))
print(f'Q1 answer: {q1} ({options_q1[q1]} kW)\n')

# =================================================================
# QUESTION 2: Total purchasing and installation costs (AUD)
# =================================================================
total_cost = 0.0
for ym in install_months:
    ns = small_systems[ym]
    nl = large_systems[ym]
    if ns == 0 and nl == 0:
        continue
    
    # Exchange rate: AUD/USD means 1 AUD = rate USD
    # To convert USD to AUD: divide by rate
    fx = exchange_rates.get(ym, None)
    if fx is None:
        print(f'  WARNING: No exchange rate for {ym}, skipping')
        continue
    
    purchase_usd = ns * SMALL_COST_USD + nl * LARGE_COST_USD
    purchase_aud = purchase_usd / fx
    install_aud = (ns + nl) * INSTALL_COST_AUD
    month_cost = purchase_aud + install_aud
    total_cost += month_cost
    print(f'  {ym[0]}-{ym[1]:02d}: s={ns} l={nl} FX={fx:.4f} purchase_USD={purchase_usd:,} purchase_AUD={purchase_aud:,.0f} install_AUD={install_aud:,} total={month_cost:,.0f}')

print(f'\nQ2: Total cost = AUD ${total_cost:,.0f} = ${total_cost/1e6:.2f}m')

options_q2 = {'A': 21.0e6, 'B': 19.5e6, 'C': 18.5e6, 'D': 17.0e6}
q2 = min(options_q2, key=lambda k: abs(options_q2[k] - total_cost))
print(f'Q2 answer: {q2} (${options_q2[q2]/1e6:.1f}m)\n')

# =================================================================
# QUESTION 3: Total electricity generated in calendar year 2016
# =================================================================
total_gen_2016 = 0.0
for target_m in range(1, 13):
    target_ym = (2016, target_m)
    days = days_in_month[target_ym]
    target_mi = month_idx(target_ym)
    month_gen = 0.0
    
    for inst_ym in install_months:
        months_since = target_mi - month_idx(inst_ym)
        if months_since < 0:
            continue
        eff = get_efficiency(months_since)
        ns = small_systems[inst_ym]
        nl = large_systems[inst_ym]
        gen = (ns * SMALL_CAPACITY_KW + nl * LARGE_CAPACITY_KW) * GENERATION_RATE * days * eff
        month_gen += gen
    
    total_gen_2016 += month_gen
    print(f'  2016-{target_m:02d}: {month_gen:,.2f} kWh')

print(f'\nQ3: Total generation 2016 = {total_gen_2016:,.0f} kWh')

options_q3 = {'A': 16154168, 'B': 16155565, 'C': 16153231, 'D': 16156943}
q3 = min(options_q3, key=lambda k: abs(options_q3[k] - total_gen_2016))
print(f'Q3 answer: {q3} ({options_q3[q3]:,} kWh)\n')

# =================================================================
# QUESTION 4: Tariff A price per kWh in February 2020
# =================================================================
tariff_a_feb2020 = get_tariff_a(2020, 2)
print(f'Q4: Tariff A in Feb 2020 = {tariff_a_feb2020:.4f} AUD/kWh')
# Feb 2020 is in fiscal year July 2019 - June 2020
# n_escalations = 2019 - 2013 = 6
# 0.25 * 1.02^6 = 0.281541...

options_q4 = {'A': 0.2760, 'B': 0.2815, 'C': 0.2822, 'D': 0.2880}
q4 = min(options_q4, key=lambda k: abs(options_q4[k] - tariff_a_feb2020))
print(f'Q4 answer: {q4} ({options_q4[q4]:.4f})\n')

# =================================================================
# QUESTION 5: Total revenue in December 2019
# =================================================================
target_ym = (2019, 12)
days = days_in_month[target_ym]
tariff_a = get_tariff_a(2019, 12)
target_mi = month_idx(target_ym)
total_rev = 0.0

print(f'Dec 2019: {days} days, Tariff A = {tariff_a:.4f}, Tariff B = {TARIFF_B}')

for inst_ym in install_months:
    months_since = target_mi - month_idx(inst_ym)
    if months_since < 0:
        continue
    eff = get_efficiency(months_since)
    ns = small_systems[inst_ym]
    nl = large_systems[inst_ym]
    if ns == 0 and nl == 0:
        continue
    
    # Generation
    gen_s = ns * SMALL_CAPACITY_KW * GENERATION_RATE * days * eff
    gen_l = nl * LARGE_CAPACITY_KW * GENERATION_RATE * days * eff
    
    # Tariff A kWh: min of house consumption and actual generation per system
    # Per small system: min(10 * days, 3 * 4 * days * eff) = min(10, 12*eff) * days
    tariff_a_per_small = min(SMALL_HOUSE_DAILY_KWH, SMALL_CAPACITY_KW * GENERATION_RATE * eff) * days
    tariff_a_per_large = min(LARGE_HOUSE_DAILY_KWH, LARGE_CAPACITY_KW * GENERATION_RATE * eff) * days
    
    tariff_a_kwh_s = ns * tariff_a_per_small
    tariff_a_kwh_l = nl * tariff_a_per_large
    
    tariff_b_kwh_s = max(0, gen_s - tariff_a_kwh_s)
    tariff_b_kwh_l = max(0, gen_l - tariff_a_kwh_l)
    
    rev = (tariff_a_kwh_s + tariff_a_kwh_l) * tariff_a + (tariff_b_kwh_s + tariff_b_kwh_l) * TARIFF_B
    total_rev += rev

print(f'\nQ5: Total revenue Dec 2019 = AUD ${total_rev:,.0f}')

options_q5 = {'A': 326705, 'B': 326731, 'C': 326624, 'D': 326756}
q5 = min(options_q5, key=lambda k: abs(options_q5[k] - total_rev))
print(f'Q5 answer: {q5} (${options_q5[q5]:,})')

In [ ]:
# =================================================================
# FINAL ANSWERS
# =================================================================

answers = {
    "question1": q1,
    "question2": q2,
    "question3": q3,
    "question4": q4,
    "question5": q5,
}

print('Final answers:')
for k, v in answers.items():
    print(f'  {k}: {v}')